<a href="https://colab.research.google.com/github/LCaravaggio/Happiness_Polarization/blob/main/Protestas_ACLED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

El problema con ACLED es que solo tiene datos de 2019 en adelante

In [1]:
from google.colab import userdata

ACLED_TOKEN = userdata.get('ACLED')

In [17]:
import requests
import pandas as pd

url = "https://acleddata.com/api/acled/read?_format=json"

def get_access_token(username, password, token_url):
    headers = {
        'Content-Type': 'application/x-www-form-urlencoded',
    }
    data = {
          'username': username,
          'password': password,
          'grant_type': "password",
          'client_id': "acled"
    }

    response = requests.post(token_url, headers=headers, data=data)

    if response.status_code == 200:
        token_data = response.json()
        return token_data['access_token']
    else:
        raise Exception(f"Failed to get access token: {response.status_code} {response.text}")

my_token = get_access_token(
    username="lcaravaggio@itba.edu.ar",
    password=ACLED_TOKEN,
    token_url="https://acleddata.com/oauth/token",
)

url = "https://acleddata.com/api/acled/read?_format=json"

In [30]:
url = "https://acleddata.com/api/acled/read?_format=json"

paises = (
    "Argentina|Bolivia|Brazil|Chile|Colombia|Costa Rica|"
    "Dominican Republic|Ecuador|El Salvador|Guatemala|Honduras|"
    "Mexico|Nicaragua|Panama|Paraguay|Peru|Uruguay|Venezuela"
)

params = {
    "country": paises,
    "year": "2000|2024",
    "year_where": "BETWEEN",
    "event_type": "Protests",

    # Protestas con intervención o fuerza excesiva
    "sub_event_type": (
        "Protest with intervention|"
        "Excessive force against protesters"
    ),

    # Además, conservar protestas con al menos una fatalidad
    "fatalities": "1",
    "fatalities_where": ">=",

    "fields": (
        "event_id_cnty|event_date|year|time_precision|"
        "country|admin1|admin2|location|"
        "event_type|sub_event_type|fatalities|notes|tags"
    ),

    "with_total": "true",
    "limit": 5000
}

response = requests.get(
    url,
    params=params,
    headers={
        "Authorization": f"Bearer {my_token}",
        "Content-Type": "application/json"
    }
)

response.raise_for_status()

data = response.json()

protestas_importantes = pd.DataFrame(data["data"])

protestas_importantes["event_date"] = pd.to_datetime(
    protestas_importantes["event_date"],
    errors="coerce"
)

protestas_primera = (
    protestas_importantes
    .sort_values(['country', 'event_date'])
    .drop_duplicates(
        subset=['country', 'year'],
        keep='first'
    )
    .reset_index(drop=True)
)
protestas_primera.to_csv('protestas.csv')

In [34]:
protestas_importantes = protestas_importantes.sort_values(
    by='event_date',
    ascending=True
).reset_index(drop=True)

protestas_importantes

,event_id_cnty,event_date,year,time_precision,event_type,sub_event_type,country,admin1,admin2,location,notes,fatalities,tags
0,VEN8468,2018-01-01,2018,2,Protests,Excessive force against protesters,Venezuela,Distrito Capital,Libertador,Caracas - Libertador,"Around 1 January 2018 (as reported), in Caraca...",1,women targeted: protesters
1,VEN7475,2018-01-09,2018,1,Protests,Excessive force against protesters,Venezuela,Portuguesa,Guanare,Guanare,"On 9 January 2018, in Guanare (Portuguesa), a ...",1,
2,HND2210,2018-01-20,2018,1,Protests,Excessive force against protesters,Honduras,Colon,Saba,Saba,"On 20 January 2018, supporters of the politica...",1,None
3,HND2228,2018-01-22,2018,1,Protests,Excessive force against protesters,Honduras,Atlantida,Arizona,Arizona,"On 22 January 2018, in Arizona, Atlantida, a g...",1,
4,VEN6993,2018-01-23,2018,1,Protests,Excessive force against protesters,Venezuela,Bolivar,Caroni,Ciudad Guayana,"On 23 January 2018, in the 25 de Marzo sector ...",1,
...,...,...,...,...,...,...,...,...,...,...,...,...,...
238,VEN23096,2024-07-29,2024,1,Protests,Excessive force against protesters,Venezuela,Zulia,San Francisco,San Francisco,"On 29 July 2024, in San Francisco (Zulia), a g...",1,Election violence; crowd size=small
239,VEN23109,2024-07-29,2024,1,Protests,Excessive force against protesters,Venezuela,Yaracuy,Pena,Yaritagua,"On 29 July 2024, in Yaritagua (Yaracuy), a dem...",1,Terrorist activity; Election violence; crowd s...
240,VEN23213,2024-07-30,2024,1,Protests,Excessive force against protesters,Venezuela,Miranda,Plaza,Guarenas,"On 30 July 2024, in Guarenas (Miranda), opposi...",1,Election violence; Terrorist activity; crowd s...
241,COL28285,2024-09-03,2024,1,Protests,Excessive force against protesters,Colombia,Valle del Cauca,Tulua,Tulua,"On 3 September 2024, in Tulua (Valle del Cauca...",1,crowd size=small
